# ITM 660 — Session 2: Classification Pipeline

This notebook walks through building a classification pipeline step by step using **scikit-learn**, **pandas**, and **seaborn**.

**Outline:**
1. Load & explore the dataset
2. Preprocess features
3. Split into train / test sets
4. Define and train classifiers
5. Evaluate with metrics and cross-validation
6. Visualize results

## Import Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

print('All imports OK')

## 1. Load & Explore the Dataset\n",
    "\n",
    "Using the **Credit Approval** dataset from the UCI ML Repository (id=27).  \n",
    "690 instances, 15 mixed features (categorical + numerical), binary target (`+` / `-`).  \n",
    "Features are anonymized (A1–A15) to protect confidentiality.\n",
    "\n",
    "To swap in a different UCI dataset, change the `id` argument in `fetch_ucirepo(id=...)`."

In [ ]:
credit_approval = fetch_ucirepo(id=27)

df = pd.concat([credit_approval.data.features, credit_approval.data.targets], axis=1)
TARGET = 'A16'

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Class distribution
print('Class distribution:')
print(df[TARGET].value_counts())

# Basic stats
df.describe()

In [ ]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
# Pairplot colored by class
sns.pairplot(df, hue=TARGET, diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Feature Pairplot', y=1.02)
plt.show()

## 2. Preprocess Features

Steps:
- Encode any **categorical** columns with `LabelEncoder`
- Impute **missing values** with column medians
- Encode the **target** if it is not numeric

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y_raw = df[TARGET].copy()

# Encode categorical features
for col in X.select_dtypes(include=['object', 'category']).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# Impute missing values
X = X.fillna(X.median(numeric_only=True))

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw.astype(str))
class_names = le.classes_

print(f'Features: {list(X.columns)}')
print(f'Classes:  {class_names}')

## 3. Train / Test Split

`stratify=y` ensures each split preserves the original class proportions.

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## 4. Define Classifiers

Each model is wrapped in a `Pipeline` so preprocessing (scaling) and the estimator are applied together.  
To add a new model, append an entry to `MODELS`.

In [ ]:
MODELS = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
    'Gradient Boosted Trees': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
}

print('Models registered:', list(MODELS.keys()))

## 5. Train & Evaluate

For each model we compute:
- **Accuracy**, **Precision**, **Recall**, and **weighted F1** on the hold-out test set
- **Cross-validated F1** (mean ± std) on the training set using `StratifiedKFold`

All multi-class metrics use `average='weighted'` to account for class imbalance.

In [ ]:
CV_FOLDS = 5
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in MODELS.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_weighted')

    results.append({
        'model':      name,
        'accuracy':   accuracy_score(y_test, y_pred),
        'precision':  precision_score(y_test, y_pred, average='weighted'),
        'recall':     recall_score(y_test, y_pred, average='weighted'),
        'f1_test':    f1_score(y_test, y_pred, average='weighted'),
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std':  cv_scores.std(),
        'y_pred':     y_pred,
    })
    print(f'{name} done')

In [ ]:
# Summary table
summary = pd.DataFrame([{
    'Model':      r['model'],
    'Accuracy':   round(r['accuracy'],   4),
    'Precision':  round(r['precision'],  4),
    'Recall':     round(r['recall'],     4),
    'F1 (test)':  round(r['f1_test'],    4),
    'CV F1 Mean': round(r['cv_f1_mean'], 4),
    'CV F1 Std':  round(r['cv_f1_std'],  4),
} for r in results])

summary.sort_values('F1 (test)', ascending=False, inplace=True)
summary

In [ ]:
# Classification report for the best model
best = max(results, key=lambda r: r['f1_test'])
print(f"Best model: {best['model']}\n")
print(classification_report(y_test, best['y_pred'], target_names=class_names))

## 6. Visualize Results

In [ ]:
# --- Bar chart: all metrics across models ---
metrics_df = pd.DataFrame({
    'Model':  [r['model'] for r in results] * 4,
    'Score':  (
        [r['accuracy']  for r in results] +
        [r['precision'] for r in results] +
        [r['recall']    for r in results] +
        [r['f1_test']   for r in results]
    ),
    'Metric': (
        ['Accuracy']    * len(results) +
        ['Precision']   * len(results) +
        ['Recall']      * len(results) +
        ['F1 Weighted'] * len(results)
    ),
})

plt.figure(figsize=(11, 4))
sns.barplot(data=metrics_df, x='Model', y='Score', hue='Metric', palette='muted')
plt.title('Model Comparison — Accuracy, Precision, Recall, F1')
plt.ylim(0, 1.05)
plt.xlabel('')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Cross-validation scores with error bars ---
cv_df = pd.DataFrame({
    'Model': [r['model'] for r in results],
    'CV F1 Mean': [r['cv_f1_mean'] for r in results],
    'CV F1 Std':  [r['cv_f1_std']  for r in results],
})

plt.figure(figsize=(7, 4))
plt.barh(
    cv_df['Model'], cv_df['CV F1 Mean'],
    xerr=cv_df['CV F1 Std'], color=sns.color_palette('muted', len(cv_df)),
    capsize=5, edgecolor='white'
)
plt.xlabel('CV Weighted F1 (mean ± std)')
plt.title(f'{CV_FOLDS}-Fold Cross-Validation F1 Scores')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# --- Confusion matrix for the best model ---
cm = confusion_matrix(y_test, best['y_pred'])

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5
)
plt.title(f"Confusion Matrix — {best['model']}")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()